# 03 Feature Baselines

Stage 6 trains traditional machine-learning baselines on compact engineered epoch-level features. The setup cell imports the reusable feature and evaluation helpers, defines repository paths, and checks whether XGBoost is available. Model selection uses the training split only through participant-level cross-validation. Validation is used once for interim evaluation and diagnostics. The held-out test split is saved as features for later final evaluation, but it is not used here for training, tuning, permutation importance, or conclusions.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
data_processed_dir = repo_root / "data" / "processed"
raw_dir = repo_root / "data" / "raw"
epoch_index_path = repo_root / "data" / "interim" / "epoch_index.csv"
split_assignments_path = repo_root / "data" / "interim" / "split_assignments.csv"
results_dir = repo_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_predictions
from src.features import FEATURE_ID_COLUMNS, build_feature_table, save_feature_tables
from src.preprocessing import TARGET_SLEEP_STAGE_LABELS

## Build Or Load Features

This section loads existing engineered feature CSVs from `data/processed/` when they are already present. If they are missing, it builds them from `data/raw/`, `data/interim/epoch_index.csv`, and `data/interim/split_assignments.csv` using `src.features.build_feature_table`, then saves separate train, validation, and test CSVs. The expected output is a small split summary with epoch and participant counts. The test feature table is created only as a saved artifact for future final evaluation.

In [2]:
feature_paths = {
    "train": data_processed_dir / "features_train.csv",
    "validation": data_processed_dir / "features_val.csv",
    "test": data_processed_dir / "features_test.csv",
}

if all(path.exists() for path in feature_paths.values()):
    train_df = pd.read_csv(feature_paths["train"], dtype={"participant_id": str})
    val_df = pd.read_csv(feature_paths["validation"], dtype={"participant_id": str})
    test_df = pd.read_csv(feature_paths["test"], dtype={"participant_id": str})
else:
    features = build_feature_table(
        raw_dir=raw_dir,
        epoch_index_path=epoch_index_path,
        split_assignments_path=split_assignments_path,
    )
    train_df = features[features["split"] == "train"].reset_index(drop=True)
    val_df = features[features["split"] == "validation"].reset_index(drop=True)
    test_df = features[features["split"] == "test"].reset_index(drop=True)
    save_feature_tables(train_df, val_df, test_df, data_processed_dir)

display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_epochs": [len(train_df), len(val_df), len(test_df)],
    "n_participants": [
        train_df["participant_id"].nunique(),
        val_df["participant_id"].nunique(),
        test_df["participant_id"].nunique(),
    ],
}))

,split,n_epochs,n_participants
0,train,55793,70
1,validation,12027,15
2,test,12269,15


## Validation Setup

This section separates identifier columns from model features, creates `X`/`y` objects for train and validation, and configures `GroupKFold` so cross-validation folds are split by `participant_id`. It also defines explicit macro-F1 scorer functions for string labels and encoded labels, avoiding binary-F1 defaults in scikit-learn. The helper at the end records validation metrics and confusion matrices in a consistent format. The expected output is no displayed table; later sections reuse these objects.

In [3]:
feature_columns = [column for column in train_df.columns if column not in FEATURE_ID_COLUMNS]
X_train = train_df[feature_columns]
y_train = train_df["label"]
groups_train = train_df["participant_id"]

X_val = val_df[feature_columns]
y_val = val_df["label"]

n_splits = min(5, groups_train.nunique())
if n_splits < 2:
    raise ValueError("Participant-level cross-validation needs at least two training participants.")

cv = GroupKFold(n_splits=n_splits)
def labeled_macro_f1_score(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        average="macro",
        labels=TARGET_SLEEP_STAGE_LABELS,
        zero_division=0,
    )

def unlabeled_macro_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

macro_f1 = make_scorer(labeled_macro_f1_score, response_method="predict")
macro_f1_unlabeled = make_scorer(unlabeled_macro_f1_score, response_method="predict")
metrics_rows = []
confusion_matrices = {}

def record_validation_result(model_name, predictions):
    metrics, matrix = evaluate_predictions(
        y_val,
        predictions,
        model_name=model_name,
        split="validation",
    )
    metrics_rows.append(metrics)
    confusion_matrices[model_name] = matrix
    return metrics, matrix

## Majority Class Baseline

This sanity-check model always predicts the most frequent training label. It is fit on the training feature table and evaluated once on the validation feature table. The expected outputs are one validation metrics row and a labeled confusion matrix, which establish the minimum useful benchmark for later models.

In [4]:
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train, y_train)
majority_predictions = majority_model.predict(X_val)
majority_metrics, majority_confusion = record_validation_result(
    "majority_class",
    majority_predictions,
)
display(pd.DataFrame([majority_metrics]))
display(majority_confusion)

,model,split,accuracy,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,majority_class,validation,0.706411,0.275983,0.0,0.0,0.0,0.706411,1.0,0.827949,0.0,0.0,0.0


,pred_Wake,pred_Non-REM,pred_REM
true_Wake,0,2110,0
true_Non-REM,0,8496,0
true_REM,0,1421,0


## Elastic-Net Multinomial Logistic Regression

This section fits a scikit-learn pipeline with median imputation, standardization, and balanced multinomial logistic regression using elastic-net regularization. `solver="saga"` supports the tuned `C` and `l1_ratio` settings; multiclass handling is left to scikit-learn's current default behavior for multinomial classification. `GridSearchCV` tunes on the training split only with participant-level cross-validation, then evaluates the selected pipeline once on validation. Expected outputs are the top CV settings, a validation metrics row, and a confusion matrix.

In [5]:
logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                solver="saga",
                class_weight="balanced",
                max_iter=10000,
                random_state=42,
            ),
        ),
    ]
)

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.0, 0.5, 1.0],
    },
    scoring=macro_f1,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
logistic_search.fit(X_train, y_train, groups=groups_train)
logistic_predictions = logistic_search.predict(X_val)
logistic_metrics, logistic_confusion = record_validation_result(
    "logistic_elasticnet",
    logistic_predictions,
)
display(pd.DataFrame(logistic_search.cv_results_).sort_values("rank_test_score").head(10))
display(pd.DataFrame([logistic_metrics]))
display(logistic_confusion)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__l1_ratio,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
6,732.060594,176.675273,0.091892,0.017998,1.00,0.0,"{'model__C': 1.0, 'model__l1_ratio': 0.0}",0.424374,0.378161,0.435529,0.471330,0.439313,0.429741,0.030147,1
7,1022.069761,304.149718,0.111851,0.022538,1.00,0.5,"{'model__C': 1.0, 'model__l1_ratio': 0.5}",0.424915,0.378517,0.434937,0.470972,0.439356,0.429739,0.029888,2
0,101.055414,21.890754,-0.007329,0.246919,0.01,0.0,"{'model__C': 0.01, 'model__l1_ratio': 0.0}",0.427993,0.377447,0.431631,0.469839,0.441257,0.429633,0.029939,3
3,314.572034,61.006955,0.079157,0.011571,0.10,0.0,"{'model__C': 0.1, 'model__l1_ratio': 0.0}",0.425876,0.379028,0.434553,0.468906,0.439771,0.429627,0.029135,4
8,1029.843661,389.489086,0.096250,0.008079,1.00,1.0,"{'model__C': 1.0, 'model__l1_ratio': 1.0}",0.425296,0.379028,0.434243,0.470668,0.438879,0.429623,0.029565,5
4,434.492735,70.919904,0.086563,0.008948,0.10,0.5,"{'model__C': 0.1, 'model__l1_ratio': 0.5}",0.426177,0.377839,0.433765,0.468882,0.440663,0.429465,0.029576,6
5,530.148534,134.999241,0.077854,0.010284,0.10,1.0,"{'model__C': 0.1, 'model__l1_ratio': 1.0}",0.425430,0.379140,0.432201,0.470133,0.440100,0.429401,0.029405,7
11,855.680327,142.565654,0.057939,0.015235,10.00,1.0,"{'model__C': 10.0, 'model__l1_ratio': 1.0}",0.423669,0.377965,0.435210,0.471328,0.438757,0.429386,0.030191,8
9,912.230837,179.771892,0.091503,0.011509,10.00,0.0,"{'model__C': 10.0, 'model__l1_ratio': 0.0}",0.423655,0.377865,0.435188,0.471379,0.438669,0.429351,0.030234,9
10,1035.440092,163.155235,0.084154,0.009426,10.00,0.5,"{'model__C': 10.0, 'model__l1_ratio': 0.5}",0.423598,0.377832,0.435135,0.471450,0.438738,0.429351,0.030270,10


,model,split,accuracy,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,logistic_elasticnet,validation,0.402095,0.374741,0.316752,0.467773,0.377727,0.796929,0.354284,0.490508,0.16342,0.590429,0.255988


,pred_Wake,pred_Non-REM,pred_REM
true_Wake,987,442,681
true_Non-REM,1872,3010,3614
true_REM,257,325,839


## XGBoost Baseline

This section trains XGBoost on all engineered features when the package is installed. Labels are encoded for XGBoost, and a modest grid over tree depth, learning rate, sampling, and regularization is tuned with participant-level training CV only. The selected model is evaluated once on validation, then validation-set permutation importance is computed as a diagnostic. Expected outputs are the top CV settings, validation metrics, a confusion matrix, and the most important features by permutation importance.

In [11]:
if XGBClassifier is None:
    print("xgboost is not installed; install it to run the XGBoost baseline.")
    xgb_search = None
    xgb_importance = pd.DataFrame()
else:
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)

    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
    )
    xgb_search = GridSearchCV(
        estimator=xgb_model,
        param_grid={
            "max_depth": [2, 3, 4],
            "learning_rate": [0.03, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_lambda": [1.0, 5.0],
            "min_child_weight": [1, 5],
        },
        scoring=macro_f1_unlabeled,
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    xgb_search.fit(X_train, y_train_encoded, groups=groups_train)
    xgb_predictions = label_encoder.inverse_transform(xgb_search.predict(X_val))
    xgb_metrics, xgb_confusion = record_validation_result(
        "xgboost_all_features",
        xgb_predictions,
    )

    validation_importance = permutation_importance(
        xgb_search.best_estimator_,
        X_val,
        y_val_encoded,
        scoring=macro_f1_unlabeled,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
    )
    xgb_importance = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance_mean": validation_importance.importances_mean,
            "importance_std": validation_importance.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)

    display(pd.DataFrame(xgb_search.cv_results_).sort_values("rank_test_score").head(10))
    display(pd.DataFrame([xgb_metrics]))
    display(xgb_confusion)
    display(xgb_importance.head(30))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_colsample_bytree,param_learning_rate,param_max_depth,param_min_child_weight,param_reg_lambda,param_subsample,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
41,9.133899,1.223340,0.156628,0.048461,0.8,0.1,4,1,1.0,1.0,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.409240,0.418683,0.454110,0.450567,0.429566,0.432433,0.017515,1
89,11.466841,2.751734,0.170991,0.074890,1.0,0.1,4,1,1.0,1.0,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.409636,0.420472,0.447151,0.455769,0.428994,0.432404,0.016945,2
47,8.418284,1.494406,0.115874,0.018266,0.8,0.1,4,5,5.0,1.0,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.410629,0.418675,0.443718,0.453705,0.429451,0.431236,0.015788,3
45,9.766218,1.945998,0.132947,0.035359,0.8,0.1,4,5,1.0,1.0,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.410626,0.421287,0.443217,0.450596,0.429762,0.431097,0.014451,4
93,8.398264,0.140239,0.131803,0.025517,1.0,0.1,4,5,1.0,1.0,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.411656,0.417276,0.449745,0.447354,0.429128,0.431032,0.015394,5
91,9.179419,3.237069,0.162564,0.075521,1.0,0.1,4,1,5.0,1.0,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.410830,0.413821,0.445770,0.453931,0.430783,0.431027,0.017006,6
94,10.645279,1.749347,0.086371,0.012423,1.0,0.1,4,5,5.0,0.8,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.413235,0.418653,0.443023,0.453654,0.425412,0.430795,0.015211,7
87,6.461918,0.119142,0.101035,0.010212,1.0,0.1,3,5,5.0,1.0,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.412882,0.418385,0.444064,0.451271,0.427183,0.430757,0.014719,8
92,10.735967,3.502563,0.128629,0.018523,1.0,0.1,4,5,1.0,0.8,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.410649,0.421302,0.440174,0.453489,0.427321,0.430587,0.014905,9
40,9.821660,1.321468,0.170846,0.032124,0.8,0.1,4,1,1.0,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.409833,0.418983,0.444236,0.451143,0.428227,0.430484,0.015361,10


,model,split,accuracy,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,xgboost_all_features,validation,0.665004,0.391442,0.372784,0.398578,0.38525,0.737553,0.842161,0.786393,0.028571,0.001407,0.002683


,pred_Wake,pred_Non-REM,pred_REM
true_Wake,841,1260,9
true_Non-REM,1282,7155,59
true_REM,133,1286,2


,feature,importance_mean,importance_std
25,ACC_Z_std,0.007007,0.001462
51,HR_max,0.004141,0.001198
34,TEMP_min,0.003189,0.000862
33,TEMP_std,0.003106,0.000872
17,ACC_Y_std,0.003087,0.000594
49,HR_std,0.002534,0.001012
38,TEMP_slope,0.001968,0.000507
52,HR_median,0.001898,0.000408
8,ACC_X_mean,0.001510,0.000660
9,ACC_X_std,0.001486,0.000758


## Interim Validation Summary

This final section combines validation metrics from the baselines, saves them to `results/stage6_validation_metrics.csv`, and displays each confusion matrix. If XGBoost ran successfully, it also saves validation-set permutation importance to `results/stage6_xgboost_validation_permutation_importance.csv`. These outputs are interim validation diagnostics, not final test-set comparisons.

In [12]:
metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df.sort_values("macro_f1", ascending=False))

metrics_output_path = results_dir / "stage6_validation_metrics.csv"
metrics_df.to_csv(metrics_output_path, index=False)
print(f"Saved validation metrics to {metrics_output_path}")

for model_name, matrix in confusion_matrices.items():
    display(model_name)
    display(matrix)

if not xgb_importance.empty:
    importance_output_path = results_dir / "stage6_xgboost_validation_permutation_importance.csv"
    xgb_importance.to_csv(importance_output_path, index=False)
    print(f"Saved validation-set permutation importance to {importance_output_path}")

,model,split,accuracy,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
2,xgboost_all_features,validation,0.665004,0.391442,0.372784,0.398578,0.385250,0.737553,0.842161,0.786393,0.028571,0.001407,0.002683
1,logistic_elasticnet,validation,0.402095,0.374741,0.316752,0.467773,0.377727,0.796929,0.354284,0.490508,0.163420,0.590429,0.255988
0,majority_class,validation,0.706411,0.275983,0.000000,0.000000,0.000000,0.706411,1.000000,0.827949,0.000000,0.000000,0.000000


Saved validation metrics to /home/manns79/dreamt-wearable-sleep-staging/results/stage6_validation_metrics.csv


'majority_class'

,pred_Wake,pred_Non-REM,pred_REM
true_Wake,0,2110,0
true_Non-REM,0,8496,0
true_REM,0,1421,0


'logistic_elasticnet'

,pred_Wake,pred_Non-REM,pred_REM
true_Wake,987,442,681
true_Non-REM,1872,3010,3614
true_REM,257,325,839


'xgboost_all_features'

,pred_Wake,pred_Non-REM,pred_REM
true_Wake,841,1260,9
true_Non-REM,1282,7155,59
true_REM,133,1286,2


Saved validation-set permutation importance to /home/manns79/dreamt-wearable-sleep-staging/results/stage6_xgboost_validation_permutation_importance.csv
